## Loading and Evaluating a Foundation Model

In [1]:
# Loading the model
from transformers import AutoModelForSequenceClassification
import torch
model = AutoModelForSequenceClassification.from_pretrained("gpt2", num_labels=2)
print(model)

/Users/olha/vscode101/Udacity/hf_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2351.41it/s, Materializing param=transformer.wte.weight]             
GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


GPT2ForSequenceClassification(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (score): Linear(in_features=768, out_features=2, bias=False)
)


In [24]:
model.config.id2label = {
    0: "ENTAILMENT",
    1: "NOT_ENTAILMENT"
}

In [26]:
model.config.label2id = {
    "ENTAILMENT": 0,
    "NOT_ENTAILMENT": 1
}

# Evaluating the model

In [48]:
# Loading the dataset
from datasets import load_dataset

splits = ["train", "validation"]
ds = {split: ds for split, ds in zip(splits, load_dataset("glue", "rte", split=splits))}
print(ds["validation"][0])


{'sentence1': 'Dana Reeve, the widow of the actor Christopher Reeve, has died of lung cancer at age 44, according to the Christopher Reeve Foundation.', 'sentence2': 'Christopher Reeve had an accident.', 'label': 1, 'idx': 0}


In [47]:
print(ds['test'][0])

{'sentence1': "Mangla was summoned after Madhumita's sister Nidhi Shukla, who was the first witness in the case.", 'sentence2': 'Shukla is related to Mangla.', 'label': -1, 'idx': 0}


In [49]:
# Loading tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(x):
    text = [
        f"Text: {s1}\nHypothesis: {s2}\nAnswer:"
        for s1, s2 in zip(x['sentence1'], x['sentence2'])
    ]
    return tokenizer(
        text,
        truncation=True
    )

# Tokenizing the data
tokenized_ds = {}
for split in splits:
    tokenized_ds[split] = ds[split].map(tokenize_function, batched=True)
print(tokenized_ds["train"][0]['input_ids'])

Map: 100%|██████████| 277/277 [00:00<00:00, 10802.12 examples/s]

[8206, 25, 1400, 18944, 286, 5674, 25034, 4062, 287, 3908, 6430, 13, 198, 49926, 313, 8497, 25, 18944, 286, 5674, 25034, 4062, 287, 3908, 13, 198, 33706, 25]


In [50]:
import torch
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader

model.config.pad_token_id = tokenizer.eos_token_id

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Remove non-tensor columns so DataCollatorWithPadding can batch cleanly
columns_to_remove = ["sentence1", "sentence2", "idx"]
test_ds = tokenized_ds["validation"].remove_columns(columns_to_remove)

data_loader = DataLoader(
    test_ds,
    batch_size=8,
    collate_fn=data_collator
)

all_logits=[]

with torch.no_grad():
    for batch in data_loader:
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )
        all_logits.append(outputs.logits)



In [51]:
print(all_logits)

[tensor([[ 4.5738, -4.4672],
        [ 4.7968, -5.0505],
        [ 5.2501, -4.9698],
        [ 4.6203, -4.7955],
        [ 4.4143, -4.3356],
        [ 4.6184, -4.0596],
        [ 4.5299, -4.1544],
        [ 4.6940, -4.1843]]), tensor([[ 4.4176, -4.4051],
        [ 4.5596, -3.9537],
        [ 4.2387, -3.9985],
        [ 3.9560, -4.0173],
        [ 5.3897, -5.3410],
        [ 3.9331, -4.2878],
        [ 4.4758, -4.7272],
        [ 4.9182, -4.3630]]), tensor([[ 4.4456, -4.8624],
        [ 4.8243, -4.2600],
        [ 4.4636, -4.7159],
        [ 4.3038, -4.2842],
        [ 3.8159, -4.0515],
        [ 3.8763, -4.3758],
        [ 4.2595, -4.8687],
        [ 4.1733, -4.0861]]), tensor([[ 4.3760, -4.5018],
        [ 3.9240, -3.9069],
        [ 4.1475, -4.3890],
        [ 4.3836, -4.3514],
        [ 4.9764, -4.8989],
        [ 4.5956, -4.7805],
        [ 5.0564, -4.9163],
        [ 5.3597, -5.3116]]), tensor([[ 4.4287, -4.7311],
        [ 5.0113, -4.7341],
        [ 4.4482, -4.5221],
        [ 4

In [52]:
preds = [logit.argmax(dim=1) for logit in all_logits]
print(preds)

[tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0,

In [53]:
predicted_labels = [
    model.config.id2label[predicted_class_id] 
    for batch in preds
    for predicted_class_id in batch.tolist()
    ]
print(predicted_labels)

['ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTAILMENT', 'ENTA

In [55]:
count_correct = 0
for index, item in enumerate(ds['validation']):
    if item['label'] == model.config.label2id[predicted_labels[index]]:
        count_correct+=1
print(f"Accuracy: {count_correct/len(predicted_labels)}")

Accuracy: 0.5270758122743683


In [ ]:
# Creating a PEFT config
# LoraConfig documentation: https://huggingface.co/docs/peft/main/en/conceptual_guides/lora
from peft import LoraConfig
config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"], # typically attention or MLP blocks
    lora_dropout=0.1,
    bias="none", # could be 'none', 'all', or 'lora_only'
    modules_to_save=["lm_head"],
    #use_rslora=True, # try with and without
)

In [56]:
# Combining the two
from peft import get_peft_model
lora_model = get_peft_model(model, config)

In [ ]:
id2lbels = {
    0: "ENTAILMENT",
    1: "NOT_ENTAILMENT"
}

label2id = {
    "ENTAILMENT": 0,
    "NOT_ENTAILMENT": 1
}

In [57]:
# Checking Trainable Parameters of a PEFT Model
lora_model.print_trainable_parameters()

trainable params: 38,892,288 || all params: 163,332,096 || trainable%: 23.8118


In [51]:
# Saving a Trained PEFT Model
lora_model.save_pretrained("gpt-lora")

In [52]:
# Loading a Saved PEFT Model
from peft import AutoPeftModelForCausalLM
lora_model = AutoPeftModelForCausalLM.from_pretrained("gpt-lora")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2711.01it/s, Materializing param=transformer.wte.weight]             


In [53]:
# Generating Text from a PEFT Model
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
inputs = tokenizer("Hello, my name is ", return_tensors="pt")
outputs = model.generate(input_ids=inputs["input_ids"], max_new_tokens=10)
print(tokenizer.batch_decode(outputs))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


['Hello, my name is _____. I am a student at the University of']
